<a href="https://colab.research.google.com/github/jkshakthi999/Real-time-Multi-Object-Colorizer/blob/main/task6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Go to Runtime > Change runtime type > T4 GPU, then run this cell again.")

!pip install -q gradio transformers
print("Libraries installed.")

PyTorch: 2.14.1+cpu
GPU available: False
Libraries installed.


In [2]:
import os, re, io, time, shutil, subprocess, tempfile, traceback
import numpy as np
import torch, torch.nn.functional as F
import cv2
from PIL import Image
from skimage import color
import matplotlib.colors as mcolors
import gradio as gr

device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------- object groups -------------------------
GROUPS = ["Vehicles", "Trees & plants", "Grass & fields", "Buildings", "Sky",
          "Roads & ground", "Water", "People", "Mountains & rocks", "Other objects"]
OTHER = len(GROUPS) - 1
# segmentation class names (ADE20K and Cityscapes) that belong to each group; everything else = "Other objects"
GROUP_NAMES = {
    "Vehicles": {"car", "bus", "truck", "van", "boat", "ship", "airplane", "minibike", "bicycle", "motorcycle", "train", "motorbike"},
    "Trees & plants": {"tree", "plant", "palm", "flower", "vegetation"},
    "Grass & fields": {"grass", "field", "terrain"},
    "Buildings": {"building", "house", "skyscraper", "tower", "hovel", "wall", "column", "bridge"},
    "Sky": {"sky"},
    "Roads & ground": {"road", "sidewalk", "path", "earth", "land", "floor", "dirt track", "runway", "sand"},
    "Water": {"water", "sea", "river", "lake", "swimming pool", "waterfall", "fountain"},
    "People": {"person", "rider"},
    "Mountains & rocks": {"mountain", "hill", "rock"},
}
# colours used only to draw the object map
DISPLAY_COLORS = [(230, 60, 60), (30, 140, 50), (140, 220, 60), (200, 150, 90), (70, 130, 235),
                  (130, 130, 140), (40, 190, 220), (245, 160, 200), (150, 100, 60), (180, 180, 60)]

# ------------------------- colour schemes (one colour per group, same order as GROUPS) -------------------------
SCHEMES = {
    "Natural": ["#B03A3A", "#3E7B3A", "#6FAF40", "#B79B7E", "#7FB6E6", "#7C7C82", "#3B88B8", "#D8A383", "#8A7A6A", "#A89480"],
    "Vivid":   ["#E02020", "#1E9A2E", "#4CD03A", "#E0A060", "#2A8CFF", "#6A6AA0", "#10A0D8", "#F0A070", "#A06A40", "#D08050"],
    "Vintage": ["#9C5B3E", "#6F6B3E", "#8D8A4A", "#B38B5D", "#B9B8A0", "#8B7D6B", "#6D8590", "#C99572", "#85705A", "#A38A6D"],
    "Pastel":  ["#E8A0A8", "#9FD3A0", "#C5E8A0", "#F0D8B8", "#B8DCF5", "#C8C8D0", "#A0D8E8", "#F5D0C0", "#D0C0B0", "#E0D0C0"],
    "Cool night": ["#4A5BA8", "#2D5A5A", "#3A7A6A", "#6A7AA8", "#3A4A8A", "#5A6078", "#2A5A9A", "#9A8AA8", "#4A5070", "#6A7090"],
}
CUSTOM = "Custom (edit the colours)"

# ------------------------- segmentation models (downloaded automatically from Hugging Face) -------------------------
SEG_MODELS = {
    "General scenes - fast (SegFormer-B0, ADE20K)": "nvidia/segformer-b0-finetuned-ade-512-512",
    "General scenes - accurate (SegFormer-B2, ADE20K)": "nvidia/segformer-b2-finetuned-ade-512-512",
    "Street scenes (SegFormer-B0, Cityscapes)": "nvidia/segformer-b0-finetuned-cityscapes-1024-1024",
}
DEFAULT_MODEL = list(SEG_MODELS)[0]

VIEWS = ["Colorized", "Original | Colorized", "Original | Colorized | Object map"]
print("Device:", device)
print("Object groups:", ", ".join(GROUPS))

Device: cpu
Object groups: Vehicles, Trees & plants, Grass & fields, Buildings, Sky, Roads & ground, Water, People, Mountains & rocks, Other objects


In [3]:
# ---------------- colour helpers ----------------
def parse_color(value):
    # colour name, hex code or rgb(...) text -> (r, g, b) integers, or None if it is not understood
    if value is None:
        return None
    s = str(value).strip().lower()
    if not s:
        return None
    m = re.match(r"rgba?\(\s*([\d.]+)\s*,\s*([\d.]+)\s*,\s*([\d.]+)", s)
    if m:
        return tuple(int(round(min(255.0, float(m.group(i))))) for i in (1, 2, 3))
    if re.fullmatch(r"[0-9a-f]{6}", s) or re.fullmatch(r"[0-9a-f]{3}", s):
        s = "#" + s
    try:
        r, g, b = mcolors.to_rgb(s.replace(" ", ""))
        return (int(round(r * 255)), int(round(g * 255)), int(round(b * 255)))
    except Exception:
        return None


def palette_to_ab(colors):
    # the 10 chosen colours -> tensor (10 x 2) of Lab (a, b) values; the brightness always comes from the video frame
    ab = []
    for g, c in zip(GROUPS, colors):
        rgb = parse_color(c)
        if rgb is None:
            raise gr.Error(f"The colour for '{g}' is not valid: {c}")
        lab = color.rgb2lab(np.array([[rgb]], dtype=np.float64) / 255.0)[0, 0]
        ab.append(lab[1:])
    return torch.tensor(np.array(ab), dtype=torch.float32, device=device)


def srgb_to_linear(c):
    return torch.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055).clamp(min=0.0) ** 2.4)


def rgb_to_L_t(rgb):
    # sRGB image 3 x H x W in [0, 1]  ->  Lab lightness 1 x H x W in 0..100
    lin = srgb_to_linear(rgb)
    Y = 0.212671 * lin[0] + 0.715160 * lin[1] + 0.072169 * lin[2]
    f = torch.where(Y > (6.0 / 29.0) ** 3, Y.clamp(min=1e-8) ** (1.0 / 3.0), Y / (3.0 * (6.0 / 29.0) ** 2) + 4.0 / 29.0)
    return (116.0 * f - 16.0)[None]


def lab_to_rgb_gpu(L, ab):
    # L: B x 1 x H x W in 0..100 ; ab: B x 2 x H x W in Lab units  ->  sRGB B x 3 x H x W in [0, 1]
    fy = (L + 16.0) / 116.0
    fx = fy + ab[:, 0:1] / 500.0
    fz = fy - ab[:, 1:2] / 200.0

    def finv(t):
        return torch.where(t > 6.0 / 29.0, t ** 3, 3.0 * (6.0 / 29.0) ** 2 * (t - 4.0 / 29.0))

    X, Y, Z = 0.95047 * finv(fx), 1.0 * finv(fy), 1.08883 * finv(fz)
    r = 3.2404542 * X - 1.5371385 * Y - 0.4985314 * Z
    g = -0.9692660 * X + 1.8760108 * Y + 0.0415560 * Z
    b = 0.0556434 * X - 0.2040259 * Y + 1.0572252 * Z
    lin = torch.cat([r, g, b], dim=1).clamp(0.0, 1.0)
    return torch.where(lin > 0.0031308, 1.055 * lin.clamp(min=1e-8) ** (1.0 / 2.4) - 0.055, 12.92 * lin)


def box_filter(x, r):
    return F.avg_pool2d(x, 2 * r + 1, stride=1, padding=r, count_include_pad=False)


def guided_filter_t(I, p, r, eps):
    # edge-aware smoothing of p guided by I (He et al.), on the GPU
    mean_I, mean_p = box_filter(I, r), box_filter(p, r)
    var_I = box_filter(I * I, r) - mean_I * mean_I
    cov_Ip = box_filter(I * p, r) - mean_I * mean_p
    a = cov_Ip / (var_I + eps)
    b = mean_p - a * mean_I
    return box_filter(a, r) * I + box_filter(b, r)


# ---------------- segmentation model ----------------
_SEG_CACHE = {}
_MEAN = None
_STD = None


def get_segmenter(model_key):
    # loads (once) the pretrained SegFormer and builds the class -> object-group matrix
    if model_key in _SEG_CACHE:
        return _SEG_CACHE[model_key]
    from transformers import SegformerForSemanticSegmentation
    net = SegformerForSemanticSegmentation.from_pretrained(SEG_MODELS[model_key]).to(device).eval()
    if device == "cuda":
        net = net.half()
    id2label = net.config.id2label
    n_cls = max(int(c) for c in id2label) + 1
    M = torch.zeros(len(GROUPS), n_cls)
    for cid, label in id2label.items():
        name = str(label).split(",")[0].strip().lower()
        g = OTHER
        for gi, gname in enumerate(GROUPS[:-1]):
            if name in GROUP_NAMES[gname]:
                g = gi
                break
        M[g, int(cid)] = 1.0
    seg = {"net": net, "M": M.to(device)}
    _SEG_CACHE[model_key] = seg
    return seg


def _round32(v):
    return max(64, int(round(v / 32.0)) * 32)


@torch.no_grad()
def colorize_frame(frame, seg, ab_groups, st, strength=1.2, temporal=0.6, refine=0.7, seg_size=512, want_map=False):
    # frame: H x W x 3 uint8 RGB. st: per-stream state dict (keeps the previous frame's object probabilities).
    # returns (colorized uint8 H x W x 3, object map uint8 H x W x 3 or None, coverage array over GROUPS)
    global _MEAN, _STD
    if _MEAN is None:
        _MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
        _STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
    t = torch.from_numpy(frame).to(device).permute(2, 0, 1).float() / 255.0
    H, W = t.shape[1:]
    L = rgb_to_L_t(t)[None]                                                           # 1 x 1 x H x W, keeps the real brightness
    gray = (0.299 * t[0] + 0.587 * t[1] + 0.114 * t[2])[None, None]

    # 1) semantic segmentation at reduced size
    k = float(seg_size) / max(H, W)
    sh, sw = _round32(H * k), _round32(W * k)
    x = F.interpolate(gray, size=(sh, sw), mode="bilinear", align_corners=False).repeat(1, 3, 1, 1)
    x = (x - _MEAN) / _STD
    if device == "cuda":
        x = x.half()
    probs = torch.softmax(seg["net"](pixel_values=x).logits.float(), dim=1)           # 1 x C x h x w
    gp = torch.einsum("gc,bchw->bghw", seg["M"], probs)                               # 1 x G x h x w (sums to 1)

    # 2) temporal smoothing (reset at scene cuts)
    thumb = F.adaptive_avg_pool2d(gray, (16, 16))
    prev = st.get("prev")
    if prev is not None and prev.shape == gp.shape and float((thumb - st["thumb"]).abs().mean()) < 0.15:
        gp = temporal * prev + (1.0 - temporal) * gp
    st["prev"], st["thumb"] = gp, thumb

    # 3) colour = probability-weighted mix of the scheme colours, then edge-aware refinement
    ab = torch.einsum("gk,bghw->bkhw", ab_groups, gp)
    ab = F.interpolate(ab, size=(H, W), mode="bilinear", align_corners=False)
    if refine > 0:
        r = max(2, int(round(max(H, W) / 150.0)))
        smooth = guided_filter_t(L / 100.0, ab / 128.0, r, 1e-3) * 128.0
        ab = (1.0 - refine) * ab + refine * smooth
    rgb = lab_to_rgb_gpu(L, ab * strength)[0]
    out = (rgb.clamp(0, 1) * 255.0).byte().permute(1, 2, 0).contiguous().cpu().numpy()

    seg_img = None
    if want_map:
        idx = gp.argmax(dim=1, keepdim=True).float()
        idx = F.interpolate(idx, size=(H, W), mode="nearest")[0, 0].long()
        dc = torch.tensor(DISPLAY_COLORS, dtype=torch.float32, device=device)
        mapimg = 0.55 * dc[idx] + 0.45 * (gray[0, 0] * 255.0)[..., None]
        seg_img = mapimg.clamp(0, 255).byte().cpu().numpy()
    cov = gp.mean(dim=(2, 3))[0].cpu().numpy()
    return out, seg_img, cov

print("Colour and segmentation code ready.")

Colour and segmentation code ready.


In [4]:
def fit_size(w, h, max_side):
    # working size: longest side <= max_side, both sides even (needed for H.264 video)
    k = min(1.0, float(max_side) / max(w, h))
    return max(2, int(w * k) // 2 * 2), max(2, int(h * k) // 2 * 2)


def compose_view(orig, colored, seg_img, view):
    if view == VIEWS[1]:
        return np.concatenate([orig, colored], axis=1)
    if view == VIEWS[2] and seg_img is not None:
        return np.concatenate([orig, colored, seg_img], axis=1)
    return colored


def coverage_text(cov, top=3):
    order = np.argsort(-cov)[:top]
    return ", ".join(f"{GROUPS[i]} {cov[i] * 100:.0f}%" for i in order if cov[i] > 0.01)


def finalize_video(tmp_path, src_path, out_path):
    # re-encode to H.264 so every browser can play it, and copy the original audio when there is one
    ff = shutil.which("ffmpeg")
    if ff is None:
        return tmp_path
    base = [ff, "-y", "-loglevel", "error", "-i", tmp_path]
    tail = ["-c:v", "libx264", "-pix_fmt", "yuv420p", "-preset", "veryfast", out_path]
    if src_path and os.path.isfile(src_path):
        cmd = base + ["-i", src_path, "-map", "0:v:0", "-map", "1:a:0?", "-c:a", "aac", "-shortest"] + tail
        if subprocess.run(cmd).returncode == 0:
            return out_path
    if subprocess.run(base + tail).returncode == 0:
        return out_path
    return tmp_path


def process_video(video_path, url, model_key, seg_size, max_side, max_seconds, view, strength, temporal, refine, *colors):
    # generator: yields (live preview frame, status text, finished video path or None) while the video is processed
    src = (url or "").strip()
    if src:
        if not re.match(r"^(https?|rtsp|rtmp)://", src, re.I):
            raise gr.Error("The video link must start with http://, https://, rtsp:// or rtmp://")
    else:
        src = video_path
    if not src:
        raise gr.Error("Please upload a video (or paste a video link) first.")
    ab_groups = palette_to_ab(colors)
    seg_size, max_side = int(seg_size), int(max_side)
    cap = cv2.VideoCapture(src)
    if not cap.isOpened():
        raise gr.Error("Could not open this video. Try an mp4 file.")
    fps_in = cap.get(cv2.CAP_PROP_FPS)
    if not fps_in or fps_in != fps_in or fps_in < 1 or fps_in > 120:
        fps_in = 25.0
    n_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    max_frames = int(max_seconds * fps_in)
    if n_total > 0:
        max_frames = min(max_frames, n_total)

    yield None, "Loading the segmentation model (first time downloads it) ...", None
    seg = get_segmenter(model_key)
    tmp_dir = tempfile.mkdtemp()
    tmp_path = os.path.join(tmp_dir, "raw.mp4")
    out_path = os.path.join(tmp_dir, "colorized.mp4")
    writer, st = None, {}
    n, t0, last_yield, preview, cov = 0, time.time(), 0.0, None, np.zeros(len(GROUPS))
    try:
        while n < max_frames:
            ok, bgr = cap.read()
            if not ok:
                break
            h0, w0 = bgr.shape[:2]
            w, h = fit_size(w0, h0, max_side)
            if (w, h) != (w0, h0):
                bgr = cv2.resize(bgr, (w, h), interpolation=cv2.INTER_AREA)
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            colored, seg_img, cov = colorize_frame(rgb, seg, ab_groups, st, strength, temporal, refine, seg_size,
                                                   want_map=(view == VIEWS[2]))
            frame_out = compose_view(rgb, colored, seg_img, view)
            if writer is None:
                writer = cv2.VideoWriter(tmp_path, cv2.VideoWriter_fourcc(*"mp4v"), fps_in,
                                         (frame_out.shape[1], frame_out.shape[0]))
            writer.write(cv2.cvtColor(frame_out, cv2.COLOR_RGB2BGR))
            n += 1
            preview = frame_out
            now = time.time()
            if now - last_yield > 0.12:
                last_yield = now
                fps = n / max(now - t0, 1e-6)
                flag = "✅ real time" if fps >= fps_in else "⚠️ slower than real time"
                tot = f"/{max_frames}" if max_frames else ""
                yield preview, (f"Frame {n}{tot} | **{fps:.1f} FPS** (video is {fps_in:.0f} FPS: {flag}) | "
                                f"objects: {coverage_text(cov)}"), None
    finally:
        cap.release()
        if writer is not None:
            writer.release()
    if n == 0:
        raise gr.Error("No frames could be read from this video.")
    fps = n / max(time.time() - t0, 1e-6)
    yield preview, f"Finishing the video file ({n} frames, {fps:.1f} FPS average) ...", None
    final = finalize_video(tmp_path, video_path if not (url or "").strip() else None, out_path)
    flag = "✅ real time" if fps >= fps_in else "⚠️ slower than real time on this GPU/settings"
    yield preview, (f"**Done.** {n} frames in {time.time() - t0:.1f} s = **{fps:.1f} FPS** on average "
                    f"(video is {fps_in:.0f} FPS: {flag}). Download the result below."), final


def live_frame(img, st, model_key, seg_size, max_side, view, strength, temporal, refine, *colors):
    # called for every webcam frame
    if img is None:
        return None, st
    if st is None:
        st = {}
    seg = get_segmenter(model_key)
    ab_groups = palette_to_ab(colors)
    h0, w0 = img.shape[:2]
    w, h = fit_size(w0, h0, int(max_side))
    rgb = np.ascontiguousarray(img[..., :3])
    if (w, h) != (w0, h0):
        rgb = cv2.resize(rgb, (w, h), interpolation=cv2.INTER_AREA)
    colored, seg_img, cov = colorize_frame(rgb, seg, ab_groups, st, strength, temporal, refine, int(seg_size),
                                           want_map=(view == VIEWS[2]))
    return compose_view(rgb, colored, seg_img, view), st


def process_image(img, model_key, seg_size, max_side, strength, refine, *colors):
    if img is None:
        raise gr.Error("Please upload an image first.")
    ab_groups = palette_to_ab(colors)
    seg = get_segmenter(model_key)
    rgb = np.asarray(img.convert("RGB"))
    h0, w0 = rgb.shape[:2]
    w, h = fit_size(w0, h0, max(int(max_side), 1080))
    if (w, h) != (w0, h0):
        rgb = cv2.resize(rgb, (w, h), interpolation=cv2.INTER_AREA)
    colored, seg_img, cov = colorize_frame(rgb, seg, ab_groups, {}, strength, 0.0, refine, int(seg_size), want_map=True)
    return colored, seg_img, {GROUPS[i]: float(cov[i]) for i in range(len(GROUPS))}

print("Processing functions ready.")

Processing functions ready.


In [5]:
try:
    seg = get_segmenter(DEFAULT_MODEL)
    test = np.zeros((360, 640, 3), np.uint8)
    test[:140] = (150, 150, 150)                 # "sky"
    test[140:260, 100:300] = (90, 90, 90)        # "building"
    test[260:] = (110, 110, 110)                 # "ground"
    ab_g = palette_to_ab(SCHEMES["Natural"])
    st = {}
    for _ in range(3):                           # warm-up
        out, mp, cov = colorize_frame(test, seg, ab_g, st, want_map=True)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    n = 20
    for _ in range(n):
        out, mp, cov = colorize_frame(test, seg, ab_g, st, want_map=True)
    fps = n / (time.time() - t0)
    print(f"Pipeline OK. Output {out.shape}, speed on a 640x360 frame: {fps:.1f} FPS ({1000 / fps:.0f} ms per frame).")
    print("Objects found in the test frame:", coverage_text(cov, 5))
    print("ffmpeg available:", shutil.which("ffmpeg") is not None)
except Exception as e:
    traceback.print_exc()
    print("\nSELF-TEST FAILED:", type(e).__name__, e)

config.json:   0%|          | 0.00/6.88k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 15.0MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

Pipeline OK. Output (360, 640, 3), speed on a 640x360 frame: 1.4 FPS (718 ms per frame).
Objects found in the test frame: Sky 43%, Buildings 43%, Other objects 10%, Roads & ground 2%, Mountains & rocks 1%
ffmpeg available: True


In [ ]:
try:
    demo.close()          # close an older app if this cell is re-run
except Exception:
    pass

HEADER = (
    "<h1 style='text-align:center; margin-bottom:0'>🎥 Real-time Multi-Object Colorizer</h1>"
    "<p style='text-align:center; margin-top:4px'><b>Semantic segmentation</b> &bull; <b>Vehicles, trees, buildings, sky, roads ...</b> &bull; "
    "<b>Colour schemes per object</b> &bull; <b>Video, webcam and images</b></p>"
)


def apply_scheme(name, *current):
    if name in SCHEMES:
        return tuple(SCHEMES[name])
    return tuple(current)


with gr.Blocks(title="Real-time Multi-Object Colorizer") as demo:
    gr.HTML(HEADER)
    with gr.Row():
        # ---------------- settings (shared by all tabs) ----------------
        with gr.Column(scale=1):
            gr.Markdown("### Settings")
            model_dd = gr.Dropdown(choices=list(SEG_MODELS), value=DEFAULT_MODEL, label="Segmentation model")
            seg_dd = gr.Dropdown(choices=[320, 512, 640, 768], value=512, label="Segmentation size (smaller = faster)")
            size_dd = gr.Dropdown(choices=[480, 640, 720, 1080], value=640, label="Output size, longest side in px (smaller = faster)")
            view_rd = gr.Radio(choices=VIEWS, value=VIEWS[1], label="Output view")
            scheme_dd = gr.Dropdown(choices=list(SCHEMES) + [CUSTOM], value="Natural", label="Colour scheme")
            with gr.Accordion("Colour of each object type (edit any of them)", open=True):
                pickers = [gr.ColorPicker(value=SCHEMES["Natural"][i], label=GROUPS[i]) for i in range(len(GROUPS))]
            strength_sl = gr.Slider(0.5, 2.0, value=1.2, step=0.1, label="Colour strength")
            temporal_sl = gr.Slider(0.0, 0.9, value=0.6, step=0.05, label="Temporal smoothing (less flicker in video)")
            refine_sl = gr.Slider(0.0, 1.0, value=0.7, step=0.05, label="Edge-aware refinement")
        # ---------------- tabs ----------------
        with gr.Column(scale=3):
            with gr.Tabs():

                with gr.Tab("🎞 Video"):
                    gr.Markdown("Upload a video file **or** paste a video link, then click **Colorize video**. "
                                "The colorized frames appear live while it is processed.")
                    vid_in = gr.Video(label="Upload a video")
                    url_tb = gr.Textbox(label="Or a video link (http, https, rtsp)", placeholder="https://.../video.mp4")
                    maxsec_sl = gr.Slider(5, 120, value=20, step=5, label="Process at most this many seconds")
                    vid_btn = gr.Button("🎨 Colorize video", variant="primary")
                    vid_status = gr.Markdown("Waiting for a video.")
                    vid_live = gr.Image(label="Live output while processing", height=420)
                    vid_out = gr.Video(label="Colorized video (download)")

                with gr.Tab("📹 Webcam (live)"):
                    gr.Markdown("Live colorization of your camera. Open the public `gradio.live` link in its own browser tab, "
                                "allow camera access, then press record/start. A small, fast setup (segmentation size 320-512, output 480-640) works best.")
                    cam_in = gr.Image(sources=["webcam"], streaming=True, type="numpy", label="Camera", height=300)
                    cam_out = gr.Image(label="Colorized live", height=420)
                    cam_state = gr.State(None)

                with gr.Tab("🖼 Image"):
                    img_in = gr.Image(type="pil", label="Upload an image", height=300)
                    img_btn = gr.Button("🎨 Colorize image", variant="primary")
                    with gr.Row():
                        img_out = gr.Image(label="Colorized", height=360)
                        img_map = gr.Image(label="Object map (what the model found)", height=360)
                    img_cov = gr.Label(label="How much of the image each object type covers", num_top_classes=len(GROUPS))

    # ---------------- events ----------------
    scheme_dd.change(apply_scheme, [scheme_dd] + pickers, pickers)

    vid_btn.click(process_video,
                  [vid_in, url_tb, model_dd, seg_dd, size_dd, maxsec_sl, view_rd, strength_sl, temporal_sl, refine_sl] + pickers,
                  [vid_live, vid_status, vid_out])

    img_btn.click(process_image, [img_in, model_dd, seg_dd, size_dd, strength_sl, refine_sl] + pickers,
                  [img_out, img_map, img_cov])

    cam_inputs = [cam_in, cam_state, model_dd, seg_dd, size_dd, view_rd, strength_sl, temporal_sl, refine_sl] + pickers
    try:
        cam_in.stream(live_frame, cam_inputs, [cam_out, cam_state], time_limit=120, stream_every=0.1)
    except TypeError:
        cam_in.stream(live_frame, cam_inputs, [cam_out, cam_state])

demo.queue()
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c8b5ebd8b9352b156d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
